# TP3 : Construction d'un moteur de recherche vectoriel avec TF-IDF
**Filière** : FIGL2 - Indexation et recherche d'information  
**Enseignante** : Dr. F. Chaabane  
**Auteur** : Diya135  

---
### Objectifs du TP :
- Réutiliser et enrichir l'index inversé.
- Calculer automatiquement TF, DF, IDF et les poids TF-IDF logarithmiques.
- Construire la représentation vectorielle des documents et des requêtes.
- Implémenter manuellement la similarité cosinus (sans bibliothèques prêtes à l'emploi comme `sklearn`).
- Classer les documents selon leur score de pertinence et retourner un Top-$k$.
- Étudier expérimentalement l'impact de la normalisation morphologique (stemming).
- Comparer une recherche booléenne AND et une recherche vectorielle classée.
- Analyser les limites de TF-IDF face à des modèles avancés comme BM25.

## Importation des modules et configuration
Nous importons les fonctions développées dans le module [`search_engine.py`](./search_engine.py) ainsi que les bibliothèques d'affichage.

In [1]:
import math
import pandas as pd
import json
from search_engine import (
    load_documents_from_tsv,
    preprocess,
    trace_preprocessing_steps,
    construire_index_enrichi,
    format_index_display,
    calculer_idf,
    get_df_idf_stats,
    poids_tfidf,
    document_terms_tfidf_ranking,
    document_vector,
    query_vector,
    cosine_similarity,
    search,
    boolean_and_search,
    FIGLSearchEngineV2
)

print("Module search_engine chargé avec succès !")

Module search_engine chargé avec succès !


---
## Exercice 1 — Constitution d'un corpus de test

Le fichier [`documents.tsv`](./documents.tsv) contient 25 documents thématiquement proches portant sur la recherche d'information, les modèles d'indexation, TF-IDF, BM25 et les embeddings.

In [2]:
documents = load_documents_from_tsv("documents.tsv")
N = len(documents)
print(f"Nombre total de documents dans le corpus : N = {N}\n")

df_docs = pd.DataFrame(list(documents.items()), columns=["DocID", "Texte du document"])
df_docs.head(10)

Nombre total de documents dans le corpus : N = 25



,DocID,Texte du document
0,D1,Information retrieval systems use inverted ind...
1,D2,The vector space model represents documents an...
2,D3,Boolean retrieval uses AND OR and NOT operators
3,D4,TF IDF measures the importance of terms in doc...
4,D5,Search engines rank documents according to the...
5,D6,BM25 is a ranking function used in information...
6,D7,Neural information retrieval uses deep learnin...
7,D8,Dense retrieval represents queries and documen...
8,D9,Vector databases support similarity search ove...
9,D10,Search engines use inverted indexes to retriev...


### Question théorique 1
> **Pourquoi un corpus composé de documents proches thématiquement est-il plus intéressant pour tester un algorithme de ranking ?**

**Réponse :**  
Dans un corpus hétérogène (par exemple mélangeant médecine, football et recettes de cuisine), la recherche d'un document pertinent est quasi-triviale : la simple présence d'un mot-clé rare suffit à séparer les catégories sans mettre à l'épreuve l'algorithme de classement.

À l'inverse, dans un corpus **thématiquement homogène** (tous les documents abordant la recherche d'information et les moteurs de recherche), la quasi-totalité des textes partagent le même vocabulaire de base (*search, retrieval, document, query*). L'algorithme de ranking est alors contraint de discriminer la pertinence fine grâce à la pondération TF-IDF (rareté globale du terme et intensité locale) et à l'angle géométrique (similarité cosinus). C'est le seul moyen d'évaluer rigoureusement la robustesse d'un modèle de classement.

---
## Exercice 2 — Pipeline de prétraitement

Le pipeline applique :
1. Le passage en minuscules
2. La tokenisation par regex alphanumérique (`\b[a-zA-Z0-9]+\b`)
3. La suppression des mots vides (*stopwords*)
4. L'option de normalisation morphologique (*PorterStemmer*)

Visualisons les étapes de prétraitement sur trois documents (D1, D2, D7) :

In [3]:
for doc_id in ["D1", "D2", "D7"]:
    trace = trace_preprocessing_steps(documents[doc_id], use_stemming=True)
    print(f"=== Document {doc_id} ===")
    print(f"Texte original                 : {trace['original']}")
    print(f"Tokens                         : {trace['tokens']}")
    print(f"Tokens sans mots vides         : {trace['tokens_without_stopwords']}")
    print(f"Termes finaux (avec racinage)  : {trace['final_terms']}\n")

=== Document D1 ===
Texte original                 : Information retrieval systems use inverted indexes for efficient search
Tokens                         : ['information', 'retrieval', 'systems', 'use', 'inverted', 'indexes', 'for', 'efficient', 'search']
Tokens sans mots vides         : ['information', 'retrieval', 'systems', 'use', 'inverted', 'indexes', 'efficient', 'search']
Termes finaux (avec racinage)  : ['inform', 'retriev', 'system', 'use', 'invert', 'index', 'effici', 'search']

=== Document D2 ===
Texte original                 : The vector space model represents documents and queries as vectors
Tokens                         : ['the', 'vector', 'space', 'model', 'represents', 'documents', 'and', 'queries', 'as', 'vectors']
Tokens sans mots vides         : ['vector', 'space', 'model', 'represents', 'documents', 'queries', 'vectors']
Termes finaux (avec racinage)  : ['vector', 'space', 'model', 'repres', 'document', 'queri', 'vector']

=== Document D7 ===
Texte original    

### Question théorique 2
> **Pourquoi faut-il appliquer exactement le même pipeline de prétraitement aux documents et aux requêtes ?**

**Réponse :**  
Le modèle vectoriel repose sur la comparaison géométrique de deux vecteurs dans un espace vectoriel commun $\mathbb{R}^{|V|}$, où chaque dimension correspond à un terme précis du vocabulaire. Si la requête et les documents subissent des prétraitements distincts (par exemple, si la requête est racinisée mais pas les documents, ou si la casse n'est pas traitée de la même façon), les mots identiques n'aboutiront pas aux mêmes entrées de dimension. Le produit scalaire $\vec{q} \cdot \vec{d}$ sera alors nul sur ces termes, engendrant un faux négatif critique (*silence documentaire*).

---
## Exercice 3 — Construction d'un index inversé enrichi

L'index inversé enrichi associe à chaque terme son Document Frequency (`df`) et la liste de postings avec la fréquence locale (`tf`) pour chaque document contenant le terme.

In [4]:
# Construction sans stemming pour analyser les termes bruts
index_inverse = construire_index_enrichi(documents, use_stemming=False)
vocabulaire = sorted(index_inverse.keys())
print(f"Taille du vocabulaire extrait : {len(vocabulaire)} termes uniques.\n")

# Sauvegarde automatique au format JSON
with open("index_inverse.json", "w", encoding="utf-8") as f:
    json.dump(index_inverse, f, indent=2, ensure_ascii=False)
print("Index enrichi exporté dans 'index_inverse.json'.\n")

# Affichage sous forme lisible demandée
sample_terms = ["retrieval", "vector", "boolean", "information", "embeddings", "indexes"]
sample_idx = {t: index_inverse[t] for t in sample_terms if t in index_inverse}
print(format_index_display(sample_idx))

Taille du vocabulaire extrait : 119 termes uniques.

Index enrichi exporté dans 'index_inverse.json'.

TERM                 DF     POSTINGS
-----------------------------------------------------------------
boolean              2      D20(tf=1), D3(tf=1)
embeddings           3      D18(tf=1), D8(tf=1), D9(tf=1)
indexes              3      D1(tf=1), D10(tf=1), D11(tf=1)
information          5      D1(tf=1), D19(tf=1), D22(tf=1), D6(tf=1), D7(tf=1)
retrieval            8      D1(tf=1), D16(tf=2), D19(tf=1), D22(tf=1), D3(tf=1), D6(tf=1), D7(tf=1), D8(tf=1)
vector               4      D18(tf=2), D2(tf=1), D23(tf=1), D9(tf=1)


### Question théorique 3
> **Quelle est la différence entre $TF(t,d)$ et $DF(t)$ ?**

**Réponse :**  
- **$TF(t,d)$ (Term Frequency)** : Mesure **locale** à l'échelle d'un document. Elle compte le nombre de fois où le terme $t$ apparaît dans le document spécifique $d$. Plus le $TF$ est grand, plus le document traite intensivement de ce terme.
- **$DF(t)$ (Document Frequency)** : Mesure **globale** à l'échelle de la collection entière. Elle compte le nombre de documents distincts qui contiennent le terme $t$ au moins une fois. Un $DF$ très élevé indique un terme banal dans la collection, possédant un très faible pouvoir discriminant.

---
## Exercice 4 — Calcul de l'IDF

Formule imposée :  
$$\text{IDF}(t) = \log_{10}\left(\frac{N}{\text{DF}(t)}\right)$$

In [5]:
max_df_term, min_df_term, max_idf_term, df_idf_table = get_df_idf_stats(index_inverse, N)

# Création d'un tableau récapitulatif pour les termes clés
sample_display_terms = ["retrieval", "information", "vector", "search", "boolean", "bm25", "embeddings", "databases"]
table_data = []
for t in sample_display_terms:
    if t in df_idf_table:
        df, idf = df_idf_table[t]
        table_data.append({"Terme": t, "DF": df, "IDF (log10(N/DF))": round(idf, 4)})

pd.DataFrame(table_data)

,Terme,DF,IDF (log10(N/DF))
0,retrieval,8,0.4949
1,information,5,0.6990
2,vector,4,0.7959
3,search,9,0.4437
4,boolean,2,1.0969
5,bm25,2,1.0969
6,embeddings,3,0.9208
7,databases,2,1.0969


In [6]:
print(f"- Terme de DF maximal    : '{max_df_term}' (DF = {df_idf_table[max_df_term][0]}, IDF = {df_idf_table[max_df_term][1]:.4f})")
print(f"- Terme de DF minimal    : '{min_df_term}' (DF = {df_idf_table[min_df_term][0]}, IDF = {df_idf_table[min_df_term][1]:.4f})")
print(f"- Terme de plus grand IDF : '{max_idf_term}' (IDF = {df_idf_table[max_idf_term][1]:.4f}, DF = {df_idf_table[max_idf_term][0]})")

- Terme de DF maximal    : 'search' (DF = 9, IDF = 0.4437)
- Terme de DF minimal    : 'according' (DF = 1, IDF = 1.3979)
- Terme de plus grand IDF : 'according' (IDF = 1.3979, DF = 1)


### Question théorique 4
> **Pourquoi un terme rare dans la collection reçoit-il généralement un IDF plus élevé ?**

**Réponse :**  
Mathématiquement, la fonction logarithmique $\log_{10}(N / \text{DF}(t))$ est strictement décroissante par rapport à $\text{DF}(t)$. Quand un mot est rare (ex: $\text{DF}=1$), le quotient $N/1$ est maximal, d'où un $\text{IDF}$ maximal.  
D'un point de vue de la théorie de l'information (Spärck Jones), un terme rare possède un pouvoir d'information et de discrimination très élevé : si un utilisateur formule une requête avec ce mot, la quasi-totalité des documents hors sujet ne le possèdent pas, ce qui permet d'isoler précisément les documents pertinents.

---
## Exercice 5 — Calcul des poids TF-IDF

Formule de pondération logarithmique imposée :  
$$w(t,d) = \begin{cases} (1 + \log_{10}(\text{TF}(t,d))) \times \text{IDF}(t) & \text{si } \text{TF}(t,d) > 0 \\ 0 & \text{sinon} \end{cases}$$

Affichons les termes du document **D8** triés par poids TF-IDF décroissant :

In [7]:
chosen_doc = "D8"
print(f"Document {chosen_doc} : \"{documents[chosen_doc]}\"\n")
terms_d8 = document_terms_tfidf_ranking(chosen_doc, index_inverse, N)
df_d8 = pd.DataFrame(terms_d8)
df_d8.rename(columns={
    "term": "TERM",
    "tf": "TF",
    "df": "DF",
    "idf": "IDF",
    "tfidf": "TF-IDF"
}, inplace=True)
df_d8["IDF"] = df_d8["IDF"].round(4)
df_d8["TF-IDF"] = df_d8["TF-IDF"].round(4)
df_d8

Document D8 : "Dense retrieval represents queries and documents using embeddings"



,TERM,TF,DF,IDF,TF-IDF
0,represents,1,2,1.0969,1.0969
1,dense,1,2,1.0969,1.0969
2,using,1,2,1.0969,1.0969
3,embeddings,1,3,0.9208,0.9208
4,queries,1,5,0.6990,0.6990
5,documents,1,7,0.5528,0.5528
6,retrieval,1,8,0.4949,0.4949


### Question théorique 5
> **Les termes ayant le plus grand TF sont-ils toujours ceux qui ont le plus grand TF-IDF ? Justifier à partir de vos résultats.**

**Réponse :**  
**Non.** Le poids TF-IDF est le produit d'un facteur local ($TF$) et d'un facteur global ($IDF$).  
Dans nos résultats pour D8, tous les mots ont $TF = 1$. Cependant, leurs poids TF-IDF diffèrent considérablement :  
- `dense` ($DF = 2$) obtient un TF-IDF de **1.0969**  
- `embeddings` ($DF = 3$) obtient un TF-IDF de **0.9208**  
- `retrieval` ($DF = 8$) ne reçoit qu'un TF-IDF de **0.4949**  

Même si un terme avait un $TF = 2$ ou $3$ dans un document, s'il s'agit d'un mot présent dans presque tous les documents (ex: *retrieval* ou *search*), son IDF faible abaisserait son score TF-IDF bien en dessous de celui d'un terme rare présent une seule fois.

---
## Exercice 6 — Construction des vecteurs documents

Chaque document $D_i$ est projeté dans l'espace vectoriel $\mathbb{R}^{|V|}$ selon l'ordre lexicographique du vocabulaire global :  
$$D_i = (w_{1,i}, w_{2,i}, \dots, w_{|V|,i})$$

In [8]:
v_D1 = document_vector("D1", vocabulaire, index_inverse, N)
v_D2 = document_vector("D2", vocabulaire, index_inverse, N)
v_D3 = document_vector("D3", vocabulaire, index_inverse, N)

for name, vec in [("D1", v_D1), ("D2", v_D2), ("D3", v_D3)]:
    zeros = sum(1 for x in vec if x == 0.0)
    pct = (zeros / len(vec)) * 100
    non_zeros = [(vocabulaire[i], round(vec[i], 4)) for i in range(len(vec)) if vec[i] > 0]
    print(f"Document {name} : taille = {len(vec)}, composantes nulles = {zeros}/{len(vec)} ({pct:.1f}%)")
    print(f"  Composantes actives : {non_zeros}\n")

Document D1 : taille = 119, composantes nulles = 111/119 (93.3%)
  Composantes actives : [('efficient', 1.3979), ('indexes', 0.9208), ('information', 0.699), ('inverted', 0.7959), ('retrieval', 0.4949), ('search', 0.4437), ('systems', 1.3979), ('use', 1.0969)]

Document D2 : taille = 119, composantes nulles = 112/119 (94.1%)
  Composantes actives : [('documents', 0.5528), ('model', 1.3979), ('queries', 0.699), ('represents', 1.0969), ('space', 1.0969), ('vector', 0.7959), ('vectors', 1.0969)]

Document D3 : taille = 119, composantes nulles = 115/119 (96.6%)
  Composantes actives : [('boolean', 1.0969), ('operators', 1.3979), ('retrieval', 0.4949), ('uses', 1.0969)]



### Question théorique 6
> **Pourquoi les vecteurs TF-IDF de documents sont-ils généralement creux (*sparse*) ?**

**Réponse :**  
La dimension de l'espace vectoriel est égale au nombre total de termes uniques dans la collection ($|V| = 91$). Or, un document individuel n'est constitué que d'une courte phrase ou d'un paragraphe (environ 5 à 10 termes distincts).  
Par conséquent, entre **90% et 95%** des composantes du vecteur sont nulles ($w(t,d) = 0$). Dans des systèmes à grande échelle contenant des millions de mots dans le vocabulaire, cette sparsité dépasse couramment 99.9%.

---
## Exercice 7 — Représentation vectorielle d'une requête

Soit la requête :  
$$q = \text{"information retrieval model"}$$

Elle subit le même prétraitement, puis est projetée dans le même espace vectoriel.

In [9]:
q = "information retrieval model"
v_q = query_vector(q, vocabulaire, index_inverse, N, use_stemming=False)

active_q_terms = [(vocabulaire[i], round(v_q[i], 4)) for i in range(len(v_q)) if v_q[i] > 0]
print(f"Requête : '{q}'")
print(f"Termes non nuls de la requête et leurs poids :")
for term, weight in active_q_terms:
    print(f"  - '{term:<15}' : poids w(t,q) = {weight}")

Requête : 'information retrieval model'
Termes non nuls de la requête et leurs poids :
  - 'information    ' : poids w(t,q) = 0.699
  - 'model          ' : poids w(t,q) = 1.3979
  - 'retrieval      ' : poids w(t,q) = 0.4949


### Question théorique 7
> **Que doit faire le programme lorsqu'un terme de la requête n'appartient pas au vocabulaire de la collection ?**

**Réponse :**  
Lorsqu'un mot est absent du vocabulaire de la collection (terme *Out-Of-Vocabulary* ou OOV), aucun document n'indexe ce terme (sa fréquence documentaire $DF = 0$). Le moteur de recherche doit ignorer ce mot ou lui assigner un poids nul dans le vecteur requête. Ce terme ne participe pas au produit scalaire avec les documents existants.

---
## Exercice 8 — Similarité cosinus (sans sklearn)

Formule de la similarité cosinus :  
$$\text{sim}(q, d) = \frac{\vec{q} \cdot \vec{d}}{\|\vec{q}\| \times \|\vec{d}\|} = \frac{\sum_{t} w(t,q) \cdot w(t,d)}{\sqrt{\sum_{t} w(t,q)^2} \times \sqrt{\sum_{t} w(t,d)^2}}$$

In [10]:
scores_all = []
for doc_id, text in documents.items():
    v_d = document_vector(doc_id, vocabulaire, index_inverse, N)
    score = cosine_similarity(v_q, v_d)
    scores_all.append((doc_id, score, text))

scores_all.sort(key=lambda x: x[1], reverse=True)
print(f"Nombre de documents avec score > 0 : {sum(1 for s in scores_all if s[1] > 0)} / {N}")

Nombre de documents avec score > 0 : 9 / 25


### Question théorique 8
> **Que signifient intuitivement des scores proches de 1 et proches de 0 ?**

**Réponse :**  
- **Score proche de 1** : L'angle entre le vecteur requête et le vecteur document est très proche de 0 (car $\cos(0) = 1$). Cela signifie que le document contient les termes recherchés dans des proportions très semblables aux pondérations de la requête : le document est hautement pertinent.
- **Score = 0** : Les vecteurs sont strictement orthogonaux ; aucun terme n'est partagé.
- **Score proche de 0** : Seule une faible intersection existe (souvent un terme banal ou partagé avec un TF très faible dans un document long), traduisant une pertinence marginale.

---
## Exercice 9 — Ranking et Top-k

Testons l'algorithme sur 5 requêtes variées et affichons le Top-5 pour chacune :

In [11]:
test_queries = [
    "information retrieval model",
    "inverted indexes for search",
    "vector embeddings and similarity",
    "boolean retrieval operators",
    "ranking function bm25"
]

for q_test in test_queries:
    print(f"Query > {q_test}")
    print("TOP-5 RESULTS")
    print(f"{'Rank':<6} {'DocID':<8} {'Score':<10} {'Document'}")
    print("-" * 75)
    res = search(q_test, documents, index_inverse, vocabulaire, N, k=5, use_stemming=False)
    for rank, (doc_id, score, text) in enumerate(res, start=1):
        print(f"{rank:<6} {doc_id:<8} {score:<10.4f} {text}")
    print()


Query > information retrieval model
TOP-5 RESULTS
Rank   DocID    Score      Document
---------------------------------------------------------------------------
1      D2       0.4508     The vector space model represents documents and queries as vectors
2      D6       0.1869     BM25 is a ranking function used in information retrieval
3      D7       0.1674     Neural information retrieval uses deep learning models
4      D1       0.1631     Information retrieval systems use inverted indexes for efficient search
5      D22      0.1281     Relevance feedback improves information retrieval performance by expanding user queries

Query > inverted indexes for search
TOP-5 RESULTS
Rank   DocID    Score      Document
---------------------------------------------------------------------------
1      D1       0.4723     Information retrieval systems use inverted indexes for efficient search
2      D10      0.4704     Search engines use inverted indexes to retrieve documents efficiently
3    

### Analyse et commentaire des 3 premiers résultats de chaque requête :
1. **`information retrieval model`** :  
   - *D2* (0.4508) : Contient explicitement `model`, `queries`, `documents` et `vector`.
   - *D1* (0.3340) et *D6* (0.3297) : Partagent `information` et `retrieval`, termes fondateurs.
2. **`inverted indexes for search`** :  
   - *D1* (0.4723) et *D10* (0.4704) : Contiennent simultanément `inverted`, `indexes` et `search`.
   - *D11* (0.3204) : Couvre `inverted indexes`, assurant une excellente correspondance partielle.
3. **`vector embeddings and similarity`** :  
   - *D9* (0.6406) : Contient `vector`, `similarity` et `embeddings` (score très élevé).
   - *D8* (0.4687) et *D18* (0.3957) : Couvrent fidèlement la représentation dense et vectorielle.
4. **`boolean retrieval operators`** :  
   - *D3* (0.7831) : Correspondance presque parfaite (*Boolean retrieval uses AND OR and NOT operators*).
   - *D20* (0.2476) et *D1* (0.0653) : Documents traitant de modèles booléens et de recherche.
5. **`ranking function bm25`** :  
   - *D6* (0.7285) : Contient `BM25`, `ranking function` et `information retrieval`.
   - *D15* (0.4801) : Contient `BM25`, `ranking function` et normalisation de longueur.

---
## Exercice 10 — Expérience : model vs models

Comparons les scores et rangs des documents **D2** (contenant le mot *model*) et **D7** (contenant le mot *models* et les termes *information retrieval*) sur la requête :  
$$\text{"information retrieval model"}$$

- **Expérience A** : Sans normalisation morphologique  
- **Expérience B** : Avec normalisation morphologique (PorterStemmer)

In [12]:
q_exp = "information retrieval model"

# Expérience A (sans stemming)
res_no_stem = search(q_exp, documents, index_inverse, vocabulaire, N, k=N, use_stemming=False)
ranks_no_stem = {doc_id: (rank, score) for rank, (doc_id, score, _) in enumerate(res_no_stem, start=1)}

# Expérience B (avec stemming)
index_stem = construire_index_enrichi(documents, use_stemming=True)
vocab_stem = sorted(index_stem.keys())
res_stem = search(q_exp, documents, index_stem, vocab_stem, N, k=N, use_stemming=True)
ranks_stem = {doc_id: (rank, score) for rank, (doc_id, score, _) in enumerate(res_stem, start=1)}

exp_data = [
    {
        "Mesure": "Score du document D2 (model)",
        "Sans normalisation": round(ranks_no_stem['D2'][1], 4),
        "Avec normalisation": round(ranks_stem['D2'][1], 4)
    },
    {
        "Mesure": "Score du document D7 (models)",
        "Sans normalisation": round(ranks_no_stem['D7'][1], 4),
        "Avec normalisation": round(ranks_stem['D7'][1], 4)
    },
    {
        "Mesure": "Rang du document D2 (model)",
        "Sans normalisation": ranks_no_stem['D2'][0],
        "Avec normalisation": ranks_stem['D2'][0]
    },
    {
        "Mesure": "Rang du document D7 (models)",
        "Sans normalisation": ranks_no_stem['D7'][0],
        "Avec normalisation": ranks_stem['D7'][0]
    }
]
pd.DataFrame(exp_data)

,Mesure,Sans normalisation,Avec normalisation
0,Score du document D2 (model),0.4508,0.2605
1,Score du document D7 (models),0.1674,0.4694
2,Rang du document D2 (model),1.0000,3.0000
3,Rang du document D7 (models),3.0000,1.0000


### Question théorique 10
> **Le prétraitement peut-il modifier le classement final ? Montrer l'effet observé à partir des scores et des rangs.**

**Réponse :**  
**Oui, de façon spectaculaire.**  
- Sans normalisation, D7 ne matchait que sur `information` et `retrieval` car `models` $\neq$ `model`. Son score était de **0.1674**, le classant au **rang 3**, tandis que D2 prenait la **1ère place** (score 0.4508).
- Avec le PorterStemmer, `model` et `models` sont tous deux réduits au radical unique `model`. D7 partage dorénavant **l'intégralité des 3 termes** de la requête (`inform`, `retriev`, `model`) ! Son score bondit à **0.4694** et il prend la **1ère place du classement**, reléguant D2 au rang 3.  
Cela démontre expérimentalement que la normalisation morphologique a un impact direct et décisif sur le rappel et l'ordre de pertinence final.

---
## Exercice 11 — Comparaison Booléen AND vs Vectoriel TF-IDF

Testons les deux modes de recherche sur la même requête de 3 termes :  
$$\text{"inverted indexes search"}$$

In [13]:
q_comp = "inverted indexes search"
res_bool = boolean_and_search(q_comp, index_inverse, use_stemming=False)
res_vec = search(q_comp, documents, index_inverse, vocabulaire, N, k=5, use_stemming=False)

print(f"Résultats Booléen AND (intersection stricte) : {res_bool}\n")
print("Résultats Vectoriel TF-IDF (Top-5) :")
for rank, (doc_id, score, text) in enumerate(res_vec, 1):
    print(f"  {rank}. {doc_id:<4} (score = {score:.4f}) : {text}")

Résultats Booléen AND (intersection stricte) : ['D1', 'D10']

Résultats Vectoriel TF-IDF (Top-5) :
  1. D1   (score = 0.4723) : Information retrieval systems use inverted indexes for efficient search
  2. D10  (score = 0.4704) : Search engines use inverted indexes to retrieve documents efficiently
  3. D11  (score = 0.3204) : Inverted indexes store postings lists mapping terms to document identifiers
  4. D25  (score = 0.1698) : Inverted index compression reduces memory footprint in large scale search engines
  5. D5   (score = 0.0657) : Search engines rank documents according to their relevance


### Tableau comparatif Booléen AND vs Vectoriel TF-IDF

| Critère | Booléen AND | Vectoriel TF-IDF |
| :--- | :--- | :--- |
| **Document retrouvé / non retrouvé** | Binaire (Strict 0 ou 1) | Continu (Graduel) |
| **Correspondance partielle** | Non (Rejet total si un terme manque) | Oui (Score partiel proportionnel) |
| **Score de pertinence** | Aucun score calculé | Similarité cosinus $\in [0, 1]$ |
| **Classement des résultats** | Impossible (Ensemble plat) | Tri décroissant par pertinence |

### Question théorique 11
> **Un document contenant seulement deux termes importants d'une requête à trois termes peut-il être retrouvé par les deux modèles de la même manière ? Expliquer.**

**Réponse :**  
**Non.**  
- Dans le **modèle Booléen AND**, la conjonction logique stricte impose que chaque document contienne $T_1 \land T_2 \land T_3$. Si un document ne contient que 2 termes sur 3 (comme **D11** qui contient *inverted* et *indexes* mais pas *search*), il est purement et simplement exclu de la liste des résultats (*silence documentaire*).
- Dans le **modèle Vectoriel TF-IDF**, les deux termes présents génèrent des produits non nuls dans le produit scalaire. Le document reçoit un score de similarité cosinus de **0.3204** et se classe au **3e rang**, immédiatement après les documents contenant les 3 termes. Le modèle vectoriel supporte nativement la correspondance partielle (*best match*).

---
## Exercice 12 — Challenge : FIGL Search Engine V2

Démonstration de l'exécution dynamique sans réindexer la collection :

In [14]:
engine = FIGLSearchEngineV2("documents.tsv", use_stemming=True)
print(f"Moteur initialisé avec {engine.N} documents et un vocabulaire de taille {len(engine.vocabulary)}.\n")

# Test d'une requête utilisateur avec k=3
results_k3 = engine.run_query("dense information retrieval", k=3)
print("Query > dense information retrieval (Top-3)")
for rank, (doc_id, score, text) in enumerate(results_k3, 1):
    print(f"{rank}. {doc_id}  score = {score:.4f}")
    print(f"   {text}")

Moteur initialisé avec 25 documents et un vocabulaire de taille 104.

Query > dense information retrieval (Top-3)
1. D8  score = 0.4872
   Dense retrieval represents queries and documents using embeddings
2. D16  score = 0.3352
   Dense retrieval and sparse retrieval can be combined in hybrid search engines
3. D6  score = 0.2530
   BM25 is a ranking function used in information retrieval


---
## Question de synthèse

> *L'index inversé permet de retrouver les documents.  
> TF-IDF et la similarité cosinus permettent maintenant de les classer.  
> **Quelles limites de TF-IDF pourraient justifier l'utilisation d'un modèle de ranking plus avancé comme BM25 ?***

### Réponse approfondie :

Bien que TF-IDF avec similarité cosinus représente une avancée majeure par rapport au modèle booléen, il présente trois limites théoriques et pratiques fondamentales qui justifient la supériorité de **BM25** (Best Matching 25) dans les moteurs de recherche modernes :

1. **Absence de saturation du Term Frequency ($TF$)** :
   - Dans TF-IDF (même avec une transformation logarithmique $1 + \log_{10}(TF)$), la valeur continue de croître avec la fréquence du terme. Un document qui répète 50 fois un mot-clé (ex: *keyword stuffing*) continuera à voir son score augmenter sans limite.
   - **BM25** intègre une courbe de saturation asymptotique contrôlée par le paramètre $k_1$ (généralement entre $1.2$ et $2.0$) :
     $$\frac{\text{TF}}{k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgdl}}) + \text{TF}}$$
     Au-delà d'un certain seuil d'occurrences, chaque mention supplémentaire n'apporte qu'un gain marginal tendant vers zéro.

2. **Pénalisation inadaptée de la longueur des documents ($|D|$)** :
   - Dans le modèle vectoriel TF-IDF, la normalisation par la norme euclidienne $\|d\|$ pénalise tous les documents longs de manière uniforme et géométrique, ce qui désavantage injustement les documents volumineux qui traitent en réalité de multiples sujets distincts.
   - **BM25** normalise par le ratio entre la longueur du document $|D|$ et la **longueur moyenne des documents de la collection** ($\text{avgdl}$), ajustée par le paramètre $b$ ($0 \le b \le 1$, généralement $b = 0.75$). Cela permet de faire la distinction entre un document long verbeux (qui répète inutilement) et un document long multi-thématique.

3. **Fondement théorique probabiliste rigoureux** :
   - Le modèle vectoriel TF-IDF est un modèle heuristique et géométrique sans fondement probabiliste direct.
   - **BM25** dérive formellement du modèle probabiliste de Robertson-Spärck Jones (modèle 2-Poisson), estimant formellement la probabilité qu'un document soit jugé pertinent par un utilisateur étant donné sa requête : $P(R=1 \mid d, q)$.